# CML 3: House Price Prediction with Linear Regression

**Scaler CML | Applied ML (Intro)**

**Goal:** fit `sklearn.linear_model.LinearRegression` on California housing (OLS solver, not from-scratch GD).

**Data:** `data/california_housing.csv` - target `median_house_value`.

## How to use this notebook

1. Run cells **top to bottom** (Restart & Run All is fine).
2. Markdown cells are the teaching track; code cells are the lab.
3. **Discuss:** / **Think:** prompts are open questions for class. Not quiz spoilers.
4. Quizzes appear inline with choices only. There is **no answer key** in this notebook.
5. Reuse CML 2 habits: **split first**, fit prep on train only, then model.

Companion notes: `Student_Notes_CML3_Linear_Regression.md`.


# Agenda

Today we connect the linear hypothesis to an end-to-end house-price Pipeline:

1. **Supervised setup** and $h_\theta(x)$ (intercept trick)
2. **Cost** $J(\theta)$ (least squares) and why we report holdout metrics
3. **GD / LMS** intuition (batch vs SGD) - theory in notes; lab uses OLS
4. **Feature scaling** inside a Pipeline
5. **Normal equation** vs iterative solvers (what sklearn actually runs)
6. **Lab:** California housing - metrics, plots, coefficients, your own house

By the end you should be able to fit a LinearRegression Pipeline safely and read MAE / RMSE / $R^2$ plus scaled coefficients without claiming causation.


# Quick theory refresh (before code)

## Hypothesis

$$
h_\theta(x) = \theta^\top x
\quad\text{with}\quad x_0 = 1
$$

One feature $\Rightarrow$ a line. Many features $\Rightarrow$ a hyperplane.

## Cost we minimize on train

$$
J(\theta) = \frac{1}{2}\sum_{i=1}^{m}\bigl(h_\theta(x^{(i)}) - y^{(i)}\bigr)^2
$$

## Solvers

| Path | Idea |
|---|---|
| Gradient descent / LMS | Iterative updates; scale features; batch vs SGD |
| Normal equation | $\theta^\ast = (X^\top X)^{-1} X^\top y$ when invertible |
| `LinearRegression` | Efficient OLS least-squares solver (not a hand-written GD loop) |

**Discuss:** Same $h_\theta$ and $J$ either way. Why call sklearn instead of coding GD for this dataset?


# Quiz 1

**With the intercept trick ($x_0 = 1$), which statement is correct?**

- $h_\theta(x)$ is a nonlinear function of $\theta$
- $h_\theta(x) = \theta^\top x$ packs intercept and slopes into one dot product
- We drop $\theta_0$ because $x_0$ is always 1
- The superscript $(i)$ means raise $x$ to the $i$-th power


## 0. Imports and load

Load California housing. We will predict `median_house_value` from a practical feature subset (same spirit as CML 2).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = pd.read_csv("data/california_housing.csv")
print(df.shape)
df.head()


## 1. Feature set and split

Same practical subset as CML 2, plus ordinal `income_band` for encoding practice.

**Think:** Why create `income_band` from `median_income` *before* the split for labels, but still insist that impute/scale/encode **fit** only on train?

**Class rule:** `train_test_split` first for model hygiene; any fitted transformer (imputer, scaler, encoder) learns from **train only**.


In [ ]:
cols = [
    "median_house_value",
    "median_income",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "ocean_proximity",
]
data = df[cols].copy()
data["income_band"] = pd.qcut(
    data["median_income"], q=3, labels=["low", "mid", "high"]
)

y = data["median_house_value"]
X = data.drop(columns=["median_house_value"])

num_cols = [
    "median_income",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
]
nom_cols = ["ocean_proximity"]
ord_cols = ["income_band"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)


## 2. Preprocess + LinearRegression pipeline

Build the CML 2-style prep, then attach OLS linear regression.

| Column type | Steps |
|---|---|
| Numeric | median impute $\rightarrow$ `StandardScaler` |
| Nominal (`ocean_proximity`) | most-frequent $\rightarrow$ one-hot |
| Ordinal (`income_band`) | most-frequent $\rightarrow$ ordered codes |
| Model | `LinearRegression` (OLS via sklearn) |

**Why scale here?** Even though OLS is closed-form, scaling:

1. Makes $|\mathrm{coef}|$ more comparable across features
2. Keeps the Pipeline ready if you later swap in `SGDRegressor` / other iterative methods

**Discuss:** Batch GD needs scaling more than OLS. Do we still scale? (Yes: interpretation + habit.)


In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[["low", "mid", "high"]])),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("nom", nominal_pipe, nom_cols),
    ("ord", ordinal_pipe, ord_cols),
])

model = Pipeline([
    ("prep", preprocess),
    ("lr", LinearRegression()),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("First 5 predictions:", np.round(y_pred[:5], 1))


# Quiz 2

**In batch gradient descent for linear regression, why update all $\theta_j$ "simultaneously"?**

- So each feature gets a different learning rate
- So every partial derivative uses the same old $\theta$ before any $\theta_j$ changes
- Because SGD forbids simultaneous updates
- To make $J$ non-convex on purpose


## 3. Holdout metrics

Training minimized squared error ($J$ / MSE). For communication on **test**:

| Metric | Read as |
|---|---|
| **MAE** | typical absolute miss in dollars (friendlier for stakeholders) |
| **RMSE** | same units as $y$; heavier penalty on large misses |
| **$R^2$** | fraction of variance explained (1 = perfect on this set) |

$$
\mathrm{MAE} = \frac{1}{m}\sum_i |\hat{y}_i - y_i|,\quad
\mathrm{RMSE} = \sqrt{\frac{1}{m}\sum_i (\hat{y}_i - y_i)^2},\quad
R^2 = 1 - \frac{\mathrm{SSE}}{\mathrm{SST}}
$$

**Think:** If MAE is about \$50k, how would you explain a single prediction to a non-ML stakeholder?


In [ ]:
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"R^2:  {r2:.4f}")


## 4. Predictions vs actual

How do test-set predictions look against the true `median_house_value`?

- **Scatter (left):** each point is one district. Closer to the diagonal $y = x$ is better.
- **Residuals (right):** error $= \hat{y} - y$ vs predicted. Want a cloud around 0 (deeper residual / assumption checks come in Session 4).
- **Table:** a few rows side by side for intuition.

**Discuss:** If residuals fan out at high predicted prices, what might that say about constant variance? (Park for Session 4.)


In [ ]:
results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": y_pred,
})
results["error"] = results["predicted"] - results["actual"]
results["abs_error"] = results["error"].abs()

print("Sample of predictions vs actual:")
print(results.head(10).round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: predicted vs actual
ax = axes[0]
ax.scatter(results["actual"], results["predicted"], alpha=0.25, s=12, color="#1971c2")
lims = [
    min(results["actual"].min(), results["predicted"].min()),
    max(results["actual"].max(), results["predicted"].max()),
]
ax.plot(lims, lims, color="#c45c26", linewidth=2, label="perfect (y = x)")
ax.set_xlabel("Actual median_house_value")
ax.set_ylabel("Predicted")
ax.set_title("Predicted vs actual (test)")
ax.legend()

# Right: residuals
ax = axes[1]
ax.scatter(results["predicted"], results["error"], alpha=0.25, s=12, color="#2a9d8f")
ax.axhline(0, color="#c45c26", linewidth=2)
ax.set_xlabel("Predicted")
ax.set_ylabel("Error (pred - actual)")
ax.set_title("Residuals (test)")

plt.tight_layout()
plt.show()

print(
    "If points hug the orange diagonal, predictions track reality. "
    "Residuals centered near 0 with no strong pattern is a good sign "
    "(deeper residual checks come in the assumptions session)."
)


## 5. Peek at coefficients

After scaling, larger $|\mathrm{coef}|$ often means a stronger **linear association** with the target in this model.

**Line to land:** association with other features held fixed in the model - **not** automatic causation.

**Think:** Before looking, which scaled feature do you expect near the top? (Often `median_income` / income band / ocean proximity.)


In [ ]:
feature_names = model.named_steps["prep"].get_feature_names_out()
coefs = model.named_steps["lr"].coef_
intercept = model.named_steps["lr"].intercept_

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

print("intercept:", round(intercept, 2))
coef_df.head(12)


# Quiz 3

**A large positive scaled coefficient on `median_income` means:**

- Raising income causes house value to rise by that many dollars (proven causation)
- In the fitted model, a 1-std increase in income associates with that change in predicted value, holding other model features fixed
- The model is overfit
- MAE must equal that coefficient


## 6. Optional: simple 1D intuition

Fit LR using only `median_income` (still via sklearn) and print slope / intercept. Plot the line against test points.

This is the geometric story from the notes: one feature $\Rightarrow$ $h_\theta$ is a **line** in the $(x, y)$ plane.

**Discuss:** Why is the multi-feature Pipeline $R^2$ usually higher than this 1D model? What did the extra features buy us?


In [ ]:
simple = LinearRegression()
simple.fit(X_train[["median_income"]], y_train)
print("slope (income):", round(simple.coef_[0], 2))
print("intercept:", round(simple.intercept_, 2))
print("test R^2:", round(simple.score(X_test[["median_income"]], y_test), 4))

# How the 1D fit looks
x_line = pd.DataFrame({
    "median_income": np.linspace(
        X_test["median_income"].min(), X_test["median_income"].max(), 100
    )
})
y_line = simple.predict(x_line)

plt.figure(figsize=(7, 5))
plt.scatter(
    X_test["median_income"], y_test, alpha=0.2, s=12, color="#1971c2", label="actual (test)"
)
plt.plot(x_line["median_income"], y_line, color="#c45c26", linewidth=2.5, label="LR fit")
plt.xlabel("median_income")
plt.ylabel("median_house_value")
plt.title("Simple LR: income -> house value")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Try your own house: enter features, get a price

Edit the values in `my_house` below, then run the cell.

`income_band` is filled automatically from `median_income` using the **train** quantile cuts (same rule as training - no test peek for bin edges here).

**Think:** Change `ocean_proximity` from `INLAND` to `NEAR OCEAN`. Does the prediction move the way the coefficients suggested?


In [ ]:
# --- edit these ---
my_house = {
    "median_income": 4.5,          # roughly tens of thousands of $ (dataset scale)
    "housing_median_age": 25.0,
    "total_rooms": 2000.0,
    "total_bedrooms": 400.0,
    "population": 1200.0,
    "households": 450.0,
    "ocean_proximity": "NEAR OCEAN",  # one of: <1H OCEAN, INLAND, ISLAND, NEAR BAY, NEAR OCEAN
}
# -------------------

# Reuse the same income_band bins learned from the full feature frame earlier
# (qcut on training incomes so we don't peek at test)
train_income = X_train["median_income"]
income_bins = pd.qcut(train_income, q=3, retbins=True, labels=["low", "mid", "high"])[1]

def income_band_from_value(income, bins=income_bins):
    # bins are edges; map to low/mid/high like training
    band = pd.cut(
        [income],
        bins=bins,
        labels=["low", "mid", "high"],
        include_lowest=True,
    )[0]
    # if somehow outside (edge float noise), clamp to nearest band
    if pd.isna(band):
        band = "low" if income <= bins[1] else "high"
    return band

my_house["income_band"] = income_band_from_value(my_house["median_income"])
my_row = pd.DataFrame([my_house])[X_train.columns]

pred_price = model.predict(my_row)[0]
print("Your input:")
print(my_row.T.rename(columns={0: "value"}))
print()
print(f"Predicted median_house_value: ${pred_price:,.0f}")
print(
    f"(For context - test MAE was about ${mae:,.0f}, so treat this as a ballpark.)"
)


# Quiz 4

**When might you prefer gradient descent over the normal equation?**

- Always, because closed form never works for linear regression
- When $n$ (number of features) is huge, or data arrives in a stream
- Only when the cost $J$ is non-convex
- When you refuse to scale features


## Takeaways

1. Hypothesis $= \theta^\top x$ (intercept via $x_0 = 1$).
2. Train by making squared error small ($J$ / OLS); report MAE / RMSE / $R^2$ on holdout.
3. Batch GD uses all $m$ points each step; SGD is cheaper and noisier. Scale features for GD and for comparable coefs.
4. Normal equation is one-shot OLS; sklearn `LinearRegression` inside a Pipeline uses an efficient solver with train-only prep.
5. Plots: predicted vs actual should hug $y = x$; residuals should sit near 0.
6. Larger $|\mathrm{coef}|$ after scaling means stronger association in the model, not proven causation.
7. Next class: assumptions and residual checks (when the least-squares story is misspecified).
